# Cosmic Event Crowdsourcing Platform: Review 2

Detects anomalous (eruptive) variable stars in crowdsourced **AAVSO** observations. The proposed model,
**RGA-Net** (Reliability-Gated Attention Network), is a small Transformer-style network whose attention is gated
by each observer's measured reliability. It is compared with four Spark MLlib models on the same windows and
the same 5 star-grouped splits.

**Stack inside this Colab VM:** Hadoop 3.3.6 (HDFS + YARN, pseudo-distributed) · Spark 3.5.9 on YARN · Spark SQL ·
Hive 3.1.3 metastore · Spark MLlib · PyTorch (RGA-Net only, on the GPU). **Every graph is a Plotly figure**, shown
inline and saved as PNG (slides) + interactive HTML. The architecture diagram is drawn with Graphviz.

Use a **GPU runtime** (Runtime → Change runtime type → T4 GPU). Step-by-step guide: `docs/COLAB_GUIDE.md`.
Outputs are written to `MyDrive/cosmic_roja/outputs/review2/` and shown at the end of the notebook.

## 0. Mount Google Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/lowkey2556/cosmic-events.git"
import os
if not os.path.isdir('/content/cosmic/.git'):
    !git clone -q "$REPO_URL" "/content/cosmic"
else:
    !cd "/content/cosmic" && git pull -q
%cd /content/cosmic
!git log --oneline | head -5

### Data folder
`DATA_ROOT` is the Drive folder that holds the AAVSO CSVs. It is searched recursively (`raw/`, `existing/`, and any
sub-folder added later). ZIPs, other files, `DOWNLOAD_LOG/` and the pipeline's own `outputs/`, `labels/` and `work/`
folders are ignored. Stars are identified by `auid`, never by file name.

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/cosmic_roja"   # <- contains raw/ (2024) and existing/ (Jan 2025); read recursively
import os
os.environ["COSMIC_DATA_ROOT"] = DATA_ROOT        # every `!python run.py ...` below reads this
assert os.path.isdir(DATA_ROOT), f"Drive folder not found: {DATA_ROOT}"
!find "$DATA_ROOT" -maxdepth 1 -type d | sort
# VSX labels fetched earlier into the old cosmic/ folder are reused (copied once), so they are not fetched again
!mkdir -p "$DATA_ROOT/labels"
!test -f "$DATA_ROOT/labels/vsx_labels.csv" || cp -v "/content/drive/MyDrive/cosmic/labels/vsx_labels.csv" "$DATA_ROOT/labels/" 2>/dev/null || true

In [ ]:
# GPU check (RGA-Net trains on the GPU if there is one; everything else uses the CPU)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - choose Runtime > Change runtime type > T4 GPU")
!nproc; free -g | head -2

## Step 0: Dataset audit (plain Python, runs before the Big Data stack)
Per file and in total: size in MB, rows, stars, observers and date range; PASS/FAIL against 50 MB, both as stored
and after removing observations repeated across overlapping downloads; per star: rows, observers, dates, coverage,
windows; class balance; skipped files; unmatched VSX types. (1 MB = 1,048,576 bytes.)

In [ ]:
!pip -q install pyyaml astropy astroquery "plotly==5.24.1" "kaleido==0.2.1"
!python run.py audit --data real

## 1. Install the stack
Java 8 comes from apt. Hadoop, Spark, Hive and Derby tarballs are cached in `MyDrive/cosmic/cache/`, so a reinstall after
a Colab reset only extracts them. The first run downloads about 1.5 GB.

In [ ]:
%%time
!bash setup/install_stack.sh

## 2. Start HDFS, YARN, the Derby network server and the Hive metastore

In [ ]:
%%time
!bash setup/start_stack.sh

In [ ]:
# Load the stack's environment (JAVA_HOME, HADOOP_HOME, SPARK_HOME, HIVE_HOME, PATH...) into this kernel,
# so every later `!command` cell can call hdfs, yarn, spark-submit and hive directly.
import os, re
env_file = open('setup/ENV_LOCATION').read().strip()
for line in open(env_file):
    m = re.match(r'export (\w+)=(.*)', line.strip())
    if m:
        os.environ[m.group(1)] = os.path.expandvars(m.group(2))
!java -version 2>&1 | head -1; hadoop version | head -1; spark-submit --version 2>&1 | grep -m1 version; hive --version 2>/dev/null | head -1

## 3. Smoke tests (each must print PASS before the pipeline runs)

In [ ]:
!bash setup/smoke_tests.sh 1   # hdfs dfs put + list

In [ ]:
!bash setup/smoke_tests.sh 2   # YARN ResourceManager up

In [ ]:
!bash setup/smoke_tests.sh 3   # SparkPi submitted to YARN

In [ ]:
!bash setup/smoke_tests.sh 4   # Hive metastore: table created and queried

In [ ]:
!bash setup/smoke_tests.sh 5   # PySpark-on-YARN Hive table read back through Hive

### Optional: open the web UIs
YARN ResourceManager is on 8088, the HDFS NameNode on 9870, and the Spark UI on 4040 while a job runs.

In [ ]:
from google.colab import output
for port in (8088, 9870):
    output.serve_kernel_port_as_window(port)

## 4. Architecture diagram

In [ ]:
!which dot >/dev/null || apt-get -qq install -y graphviz   # Graphviz draws the diagram
!python run.py architecture --data real

## 5. Step 1: Ingestion and storage (HDFS, Spark, Parquet, Hive)
`hdfs dfs -put` to `/cosmic/raw/` → explicit-schema read → Parquet + Snappy partitioned by `obs_year`/`sky_region` →
Hive external table `observations_raw` → VSX labels → partition-pruning test → dataset summary.

In [ ]:
!python run.py ingest --data real

In [ ]:
# The raw data in HDFS, and the partitions as Hive itself sees them
!hdfs dfs -ls -R -h "/cosmic/raw" | head -40
!hdfs dfs -du -s -h "/cosmic/raw" "/cosmic/warehouse/observations_raw" "/cosmic/warehouse/observations_flat"
!hive -S -e "SHOW PARTITIONS cosmic.observations_raw;" 2>/dev/null

## 6. Step 2: Preprocessing in Spark SQL
Deduplication → null keys → comment codes → fainter-than split → **thinning** (≤ 10 points per observer per star per band
per night) → band normalisation. Rows are counted after every step, in total and per star.

In [ ]:
!python run.py preprocess --data real

## 7. Step 3: Observer agreement score A_k

In [ ]:
!python run.py score --data real

## 8. Step 4: Per-star features (used by K-Means)

In [ ]:
!python run.py features --data real

## 9. Step 5: Windowing in Spark (the samples for every model)
100-day windows (stride 50 days, ≥ 20 observations) → per-window ML features (Spark SQL + Lomb–Scargle in
applyInPandas) and RGA-Net token sequences (L = 200) → Parquet on HDFS + Hive tables → 5 star-grouped 70/15/15
splits → export to `work/windows_seq.npz` for PyTorch. Also writes `star_attrition.csv` (every star: used or not, and why).

In [ ]:
!python run.py windows --data real

In [ ]:
# The window tables in Hive
!hive -S -e "USE cosmic; SHOW TABLES;" 2>/dev/null
!hive -S -e "SELECT label, COUNT(*) FROM cosmic.window_features GROUP BY label;" 2>/dev/null

## 10. Step 6: K-Means on per-star features (supporting analysis)

In [ ]:
!python run.py cluster --data real

## 11. Step 7: The four ML models (Spark MLlib) on the windows
Naive Bayes (gaussian), Decision Tree, Random Forest, Logistic Regression; 5 splits; class weights via `weightCol`;
scaling fitted on train; grids tuned on validation; with and without observer features.

In [ ]:
%%time
!python run.py classify --data real

## 12. Step 8: RGA-Net and its ablations (PyTorch, GPU)
Full model, no-gate (λ = 0) and observer-blind, each on the same 5 splits. Early stopping on validation PR-AUC.

In [ ]:
%%time
!python run.py rganet --data real

## 13. Step 9: Comparison (tables, figures, COMPARISON.md)

In [ ]:
!python run.py compare --data real

## 14. Step 10: Scalability (YARN with 1, 2 and 4 executors, 3 runs each)
Re-runs preprocessing + windowing 9 times, so it takes a while. Run it last.

In [ ]:
%%time
!python run.py scalability --data real

# Review 2 results (all from the real data above)

In [ ]:
import os
import pandas as pd
import plotly.io as pio
from IPython.display import Image, Markdown, display
OUT = '/content/drive/MyDrive/cosmic_roja/outputs/review2'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 120)
def csv(name, title=None):
    display(Markdown(f'**{title or name}**'))
    path = f'{OUT}/{name}'
    display(pd.read_csv(path)) if os.path.exists(path) else display(Markdown(f'*{name}: not produced yet*'))
def fig(name):
    # show a saved Plotly figure inline (interactive); its PNG and HTML copies are next to it in OUT
    path = f'{OUT}/{name}.json'
    pio.read_json(path).show() if os.path.exists(path) else display(Markdown(f'*{name}: not produced yet*'))
def png(name, width=900):
    display(Image(f'{OUT}/{name}', width=width))
if os.path.exists(f'{OUT}/figure_export_log.csv'):
    log = pd.read_csv(f'{OUT}/figure_export_log.csv')
    bad = log[~log['png'].astype(bool)]
    print('PNG export: all OK' if bad.empty else f'PNG export FAILED for {len(bad)} figures (HTML kept):')
    if not bad.empty: display(bad)

## Dataset

In [ ]:
csv('audit_summary.csv', 'Dataset audit: summary (50 MB check)')
csv('dataset_summary.csv', 'Dataset summary after ingestion')
csv('audit_class_balance.csv', 'Class balance'); csv('audit_skipped_files.csv', 'Skipped files (bad header)')
csv('audit_unmatched_vsx_types.csv', 'Unmatched VSX types')
csv('rows_per_band.csv', 'Rows per band'); csv('rows_per_type.csv', 'Rows per observation type')
fig('fig_obs_per_star')

## Architecture

In [ ]:
png('architecture.png', 1100)   # Graphviz PNG (SVG and DOT source saved next to it)

## Preprocessing results

In [ ]:
csv('preprocess_row_counts.csv', 'Rows after each step (thinning is step 5)'); fig('fig_preprocessing_funnel')
csv('partition_pruning.csv', 'Partition pruning: partitioned vs unpartitioned'); fig('fig_partition_pruning')
csv('lightcurve_star.csv', 'Light-curve star'); fig('fig_lightcurve_before_after')
csv('observer_score_summary.csv', 'Observer scores: summary'); fig('fig_observer_score_hist')
csv('observer_top_bottom10.csv', 'Top and bottom 10 observers by agreement score')
csv('spearman_score_vs_uncertainty.csv', 'Spearman: score vs self-reported uncertainty')

## Windows, splits and star attrition

In [ ]:
csv('window_summary.csv', 'Windows'); csv('window_counts_by_class.csv', 'Windows per class')
csv('window_counts_by_split.csv', 'Windows per split and class (every seed)')
att = pd.read_csv(f'{OUT}/star_attrition.csv')
print(att['status'].value_counts().to_string())
display(att[att['status'] != 'used'][['auid', 'name', 'vsx_type', 'status', 'reason']])

## Main result: RGA-Net vs the four ML models (5 star-grouped splits)

In [ ]:
csv('comparison_table.csv', 'Test windows, mean ± std over 5 splits (positive class = anomalous)')
csv('comparison_table_star_level.csv', 'Test stars (mean window probability), mean ± std over 5 splits')
fig('fig_comparison_metrics'); fig('fig_comparison_metrics_star_level')
fig('fig_roc_pr_curves'); fig('fig_confusion_matrices')

## RGA-Net: training curves, ablation, learned λ

In [ ]:
fig('fig_rganet_training_curves')
csv('ablation_table.csv', 'Ablation, test windows (mean ± std)'); fig('fig_ablation')
csv('rganet_learned_lambda.csv', 'Learned gate strengths λ (full model)')

## Supplementary: ML models with vs without observer features

In [ ]:
csv('ml_observer_features_table.csv', 'ML models, test windows (mean ± std)'); csv('ml_selected_params.csv', 'Chosen settings and training time')

## COMPARISON.md (generated from the numbers above)

In [ ]:
p = f'{OUT}/COMPARISON.md'
display(Markdown(open(p, encoding='utf-8').read())) if os.path.exists(p) else print('run the compare step first')

## Scalability

In [ ]:
csv('scalability.csv', 'Preprocessing + windowing through YARN'); fig('fig_scalability')

## Supporting: K-Means

In [ ]:
csv('kmeans_silhouette.csv', 'K-Means silhouette'); fig('fig_kmeans_silhouette')
csv('kmeans_crosstab_label.csv', 'Cluster vs label')

In [ ]:
!ls -la "/content/drive/MyDrive/cosmic_roja/outputs/review2"